# Red Neuronal Artificial - EMNIST Digits
**Librerías permitidas: numpy, tensorflow, keras**

Flujo del profesor:
1. Cargar datos → 2. Arquitectura → 3. Compilar → 4. Entrenar → 5. Predecir → 6. Error de testeo

## 1. Importar librerías (solo las permitidas)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow:', tf.__version__)
print('Keras:', keras.__version__)

## 2. Cargar dataset EMNIST Digits
Formato CSV: columna 0 = etiqueta, columnas 1-784 = píxeles (imagen 28×28)

In [ ]:
RUTA = r'C:\Users\ayuqu\Downloads\DB'

print('Cargando datos de entrenamiento...')
datos_train = np.genfromtxt(RUTA + r'\emnist-digits-train.csv', delimiter=',')
print(f'Train cargado: {datos_train.shape}')

print('Cargando datos de testeo...')
datos_test = np.genfromtxt(RUTA + r'\emnist-digits-test.csv', delimiter=',')
print(f'Test cargado:  {datos_test.shape}')

In [ ]:
# Separar etiquetas y caracteristicas
y_train = datos_train[:, 0].astype(int)    # columna 0 = etiqueta
X_train = datos_train[:, 1:].astype('float32')  # columnas 1-784 = pixeles

y_test = datos_test[:, 0].astype(int)
X_test = datos_test[:, 1:].astype('float32')

n_features = X_train.shape[1]  # 784

print(f'X_train: {X_train.shape}  |  y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape}   |  y_test:  {y_test.shape}')
print(f'n_features: {n_features}')
print(f'Clases: {np.unique(y_train)}')

In [ ]:
# Normalizar pixeles al rango [0, 1]
X_train = X_train / 255.0
X_test  = X_test  / 255.0

print('Normalizacion completa')
print(f'Min: {X_train.min()}  Max: {X_train.max()}')

## 3. Construir el modelo (igual que el profesor)

```
Input(784) → Dense(25, relu) → Dense(15, relu) → Dense(5, relu) → Dense(10, softmax)
```
- **relu** en capas ocultas (evita desvanecimiento de gradiente)
- **softmax** en salida → 10 clases (dígitos 0-9)
- Solo la primera capa lleva `input_shape` (como dijo el profesor)

In [ ]:
# Instanciar modelo secuencial
model = keras.Sequential()

# n_features = 784 (tenemos 784 variables/columnas de pixeles)
# Solo la primera capa viene con input_shape - como explico el profesor
model.add(layers.Dense(25, activation='relu', input_shape=(n_features,)))
model.add(layers.Dense(15, activation='relu'))
model.add(layers.Dense(5,  activation='relu'))

# Capa de salida: 10 neuronas, una por cada digito (0-9)
model.add(layers.Dense(10, activation='softmax'))

## 4. Compilar el modelo

In [ ]:
# Como explico el profesor:
# - loss: entropia cruzada
# - optimizer: adam (tasa de aprendizaje adaptativa)
# - metrics: accuracy (exactitud)
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Ver resumen del modelo - model.summary() como en clase
model.summary()

## 5. Entrenar el modelo (30 épocas como el profesor)

In [ ]:
# model.fit - 30 epocas como menciono el profesor
# En cada epoca se muestra la exactitud
history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=64,
    verbose=1
)

print('\nEntrenamiento completado!')

In [ ]:
# Ver exactitud por epoca (como mostro el profesor)
print('\nExactitud por epoca (entrenamiento):')
for epoca, acc in enumerate(history.history['accuracy'], 1):
    print(f'  Epoca {epoca:2d}: accuracy = {acc:.4f}')

## 6. Predicciones y error de testeo
Como dijo el profesor: guardar datos de testeo, correr `model.predict()` **solo con testeo**, comparar predicciones vs etiquetas verdaderas.

In [ ]:
# model.predict solo con los datos de testeo (guardados aparte)
y_pred_proba = model.predict(X_test)

# La clase predicha es la neurona con mayor probabilidad
y_pred = np.argmax(y_pred_proba, axis=1)

print('Primeras 10 predicciones:', y_pred[:10])
print('Primeras 10 reales:      ', y_test[:10])

In [ ]:
# Error de testeo - como explico el profesor
# Comparar predicciones vs etiquetas verdaderas
correctas = np.sum(y_pred == y_test)
total     = len(y_test)

exactitud_testeo = correctas / total
error_testeo     = 1 - exactitud_testeo

print('=' * 50)
print('   RESULTADO EN DATOS DE TESTEO')
print('=' * 50)
print(f'  Muestras testeo:   {total}')
print(f'  Correctas:         {correctas}')
print(f'  Exactitud:         {exactitud_testeo:.4f}  ({exactitud_testeo*100:.2f}%)')
print(f'  Error de testeo:   {error_testeo:.4f}')

# El profesor dijo: el error no puede ser mucho mayor que 0.10
if error_testeo <= 0.10:
    print('\n  La arquitectura es adecuada (error <= 0.10)')
else:
    print('\n  La arquitectura necesita ajustes (error > 0.10)')

In [ ]:
# Tambien con model.evaluate para confirmar
loss_test, acc_test = model.evaluate(X_test, y_test, verbose=0)
print(f'Loss (entropia cruzada): {loss_test:.4f}')
print(f'Exactitud (evaluate):    {acc_test:.4f}')

## 7. Resumen del deber

| Concepto del profesor | Código |
|---|---|
| Modelo secuencial | `keras.Sequential()` |
| Agregar capas | `model.add(layers.Dense(...))` |
| Solo 1ra capa con input_shape | `input_shape=(n_features,)` |
| Activación capas ocultas | `relu` |
| Compilar con entropía cruzada | `sparse_categorical_crossentropy` |
| Optimizador Adam | `optimizer='adam'` |
| Métrica exactitud | `metrics=['accuracy']` |
| Ver parámetros | `model.summary()` |
| Entrenar 30 épocas | `model.fit(epochs=30)` |
| Predecir solo con testeo | `model.predict(X_test)` |
| Error de testeo bajo | `error_testeo <= 0.10` |